# AI Job Market: Data Quality Audit

This notebook audits data types, missing values, duplicate records,
and invalid value ranges before further analysis.

In [ ]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("../ai_jobs_market_2025_2026.csv")
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
df.head()

## 1. Data types

We inspect whether each column was loaded with an appropriate type.

In [ ]:
df.info()

In [ ]:
dtype_report = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values
})

dtype_report

## 2. Missing values

We check both standard missing values and blank strings because empty
text is not always recognized as NaN.

In [ ]:
missing_report = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percentage": (
        df.isna().mean().mul(100).round(2)
    )
}).sort_values(
    "missing_count",
    ascending=False
)

missing_report

In [ ]:
print("Total missing cells:", df.isna().sum().sum())
print(
    "Columns with missing values:",
    (df.isna().sum() > 0).sum()
)

In [ ]:
text_columns = df.select_dtypes(include="object").columns

blank_string_report = (
    df[text_columns]
    .apply(lambda column: column.str.strip().eq("").sum())
    .sort_values(ascending=False)
)

blank_string_report

In [ ]:
print(
    "Total blank strings:",
    blank_string_report.sum()
)

### Observation

The dataset contains no standard missing values and no blank strings
in text columns. No imputation is required for the current data.

## 3. Duplicate records

We check exact duplicate rows, duplicate job IDs, and potentially
duplicated job postings.

In [ ]:
exact_duplicate_count = df.duplicated().sum()

print("Exact duplicate rows:", exact_duplicate_count)

In [ ]:
duplicate_job_id_count = df["job_id"].duplicated().sum()

print("Duplicate job IDs:", duplicate_job_id_count)
print("Unique job IDs:", df["job_id"].nunique())
print("Total rows:", len(df))

In [ ]:
duplicate_check_columns = [
    "job_title",
    "experience_level",
    "annual_salary_usd",
    "city",
    "country",
    "required_skills"
]

potential_duplicates = df[
    df.duplicated(
        subset=duplicate_check_columns,
        keep=False
    )
].sort_values(duplicate_check_columns)

print(
    "Potentially duplicated rows:",
    len(potential_duplicates)
)

potential_duplicates.head(10)

### Observation

The dataset contains no exact duplicate rows, duplicate job IDs, or
potential duplicates based on the selected job attributes. All 1,500
job IDs are unique.

In [ ]:
range_columns = [
    "years_of_experience",
    "annual_salary_usd",
    "salary_min_usd",
    "salary_max_usd",
    "demand_score",
    "benefits_score_10",
    "posting_year",
    "posting_month"
]

df[range_columns].describe().T

In [ ]:
quality_checks = pd.Series({
    "negative_experience": (
        df["years_of_experience"] < 0
    ).sum(),

    "non_positive_annual_salary": (
        df["annual_salary_usd"] <= 0
    ).sum(),

    "salary_min_above_max": (
        df["salary_min_usd"] > df["salary_max_usd"]
    ).sum(),

    "annual_salary_outside_range": (
        ~df["annual_salary_usd"].between(
            df["salary_min_usd"],
            df["salary_max_usd"]
        )
    ).sum(),

    "invalid_month": (
        ~df["posting_month"].between(1, 12)
    ).sum(),

    "invalid_demand_score": (
        ~df["demand_score"].between(0, 100)
    ).sum(),

    "invalid_benefits_score": (
        ~df["benefits_score_10"].between(0, 10)
    ).sum()
}).to_frame("invalid_rows")

quality_checks

In [ ]:
binary_columns = [
    "is_senior",
    "is_remote_friendly",
    "is_llm_role"
]

binary_value_report = {
    column: sorted(df[column].unique())
    for column in binary_columns
}

binary_value_report

## 4. Cleaning decisions

We preserve the raw DataFrame and create a cleaned copy with clearer
data types for binary indicator columns.

In [ ]:
clean_df = df.copy()

clean_df[binary_columns] = (
    clean_df[binary_columns].astype(bool)
)

clean_df[binary_columns].dtypes

In [ ]:
print("Raw shape:", df.shape)
print("Clean shape:", clean_df.shape)

## Conclusion

- The dataset contains 1,500 rows and 25 columns.
- No standard missing values or blank strings were found.
- No exact duplicates, duplicate job IDs, or potential duplicate
  postings were detected.
- Salary ranges, months, scores, and experience values passed the
  defined validity checks.
- Binary indicator columns contained only 0 and 1 and were converted
  to Boolean values in a cleaned copy.
- No rows were removed and no missing-value imputation was required.

The dataset passes the current quality checks and is suitable for the
next stage of analysis. These checks validate internal consistency,
but they do not prove that the dataset is unbiased or representative
of the entire AI job market.